# Session 4.1: Cross-Validation Deep Dive
## Module 4: Model Evaluation & Optimization

**Learning Objectives:**
- Understand why cross-validation is better than a single train/test split
- Implement K-Fold cross-validation for reliable performance estimates
- Use Stratified K-Fold for imbalanced datasets
- Apply Time Series cross-validation for temporal data
- Compare different CV strategies and understand when to use each
- Interpret CV results and standard deviation

**Why Cross-Validation Matters:**
A single train/test split might give you lucky or unlucky results depending on which data points end up in each set. Cross-validation uses multiple splits to give you a more reliable estimate of model performance with mean ± standard deviation.

**What You'll Build:**
- Compare train/test split vs K-Fold CV on real data
- Show how Stratified K-Fold helps with imbalanced data
- Demonstrate time series CV preventing data leakage
- Visualize performance variability across folds

## Step 1: Import Libraries and Setup

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Machine learning
from sklearn.datasets import make_classification
from sklearn.model_selection import (
    train_test_split, 
    cross_val_score,
    KFold, 
    StratifiedKFold, 
    TimeSeriesSplit,
    cross_validate
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plotting setup
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✅ Libraries imported successfully!")
print(f"Random seed: {RANDOM_STATE}")

## Step 2: Create Datasets for Different CV Scenarios

We'll create three datasets:
1. **Balanced dataset** - for regular K-Fold CV
2. **Imbalanced dataset** - to show why Stratified K-Fold is needed
3. **Time-series dataset** - to demonstrate Time Series CV

In [ ]:
print("CREATING DATASETS")
print("="*60)

# 1. Balanced dataset for binary classification
X_balanced, y_balanced = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=15,
    n_redundant=5,
    weights=[0.5, 0.5],  # Balanced classes
    random_state=RANDOM_STATE
)

print("1. Balanced Dataset:")
print(f"   Total samples: {len(y_balanced)}")
print(f"   Class 0: {(y_balanced==0).sum()} ({(y_balanced==0).sum()/len(y_balanced)*100:.1f}%)")
print(f"   Class 1: {(y_balanced==1).sum()} ({(y_balanced==1).sum()/len(y_balanced)*100:.1f}%)")

# 2. Imbalanced dataset (10% minority class)
X_imbalanced, y_imbalanced = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=15,
    n_redundant=5,
    weights=[0.9, 0.1],  # 10% minority class
    random_state=RANDOM_STATE
)

print("\n2. Imbalanced Dataset:")
print(f"   Total samples: {len(y_imbalanced)}")
print(f"   Class 0: {(y_imbalanced==0).sum()} ({(y_imbalanced==0).sum()/len(y_imbalanced)*100:.1f}%)")
print(f"   Class 1: {(y_imbalanced==1).sum()} ({(y_imbalanced==1).sum()/len(y_imbalanced)*100:.1f}%)")

# 3. Time-series dataset
n_samples = 1000
time_index = pd.date_range('2023-01-01', periods=n_samples, freq='D')
X_timeseries = np.random.randn(n_samples, 10)

# Add trend component
trend = np.linspace(0, 10, n_samples).reshape(-1, 1)
X_timeseries[:, 0] += trend.flatten()

# Target depends on trend (simulating time-dependent pattern)
y_timeseries = (X_timeseries[:, 0] + np.random.randn(n_samples)*2 > 5).astype(int)

print("\n3. Time-Series Dataset:")
print(f"   Total samples: {len(y_timeseries)}")
print(f"   Date range: {time_index[0]} to {time_index[-1]}")
print(f"   Has temporal trend: YES")

print("\n✅ All datasets created!")

## Step 3: Single Train/Test Split (The Problem)

Let's first see the problem with a single train/test split - the results can vary significantly depending on how you split the data.

> **💡 AI Assistant Prompt:**
> 
> "Explain why a single train/test split might give unreliable performance estimates. Include code examples showing how different random seeds can produce different results."

In [ ]:
print("SINGLE TRAIN/TEST SPLIT - Variability Demo")
print("="*60)

# Try 10 different random splits
split_scores = []

for seed in range(10):
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X_balanced, y_balanced, test_size=0.2, random_state=seed
    )
    
    # Train model
    model = LogisticRegression(random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    
    # Evaluate
    score = model.score(X_test, y_test)
    split_scores.append(score)
    print(f"Split {seed+1}: Accuracy = {score:.4f}")

print(f"\nMean accuracy: {np.mean(split_scores):.4f}")
print(f"Std deviation: {np.std(split_scores):.4f}")
print(f"Range: [{np.min(split_scores):.4f}, {np.max(split_scores):.4f}]")

# Visualize variability
plt.figure(figsize=(10, 5))
plt.plot(range(1, 11), split_scores, 'o-', linewidth=2, markersize=8)
plt.axhline(y=np.mean(split_scores), color='r', linestyle='--', 
            label=f'Mean: {np.mean(split_scores):.4f}')
plt.fill_between(range(1, 11), 
                 np.mean(split_scores) - np.std(split_scores),
                 np.mean(split_scores) + np.std(split_scores),
                 alpha=0.2, color='red')
plt.xlabel('Different Random Split', fontsize=12)
plt.ylabel('Test Accuracy', fontsize=12)
plt.title('Performance Variability with Different Train/Test Splits', 
          fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n⚠️ PROBLEM: Accuracy varies based on how we split!")
print("   Which score should we report? This is where CV helps.")

## Step 4: K-Fold Cross-Validation (The Solution)

K-Fold CV divides data into K equal parts. Each part serves as test set once while the others train the model. This gives us K performance scores to average.

**How 5-Fold CV Works:**
- Split data into 5 equal parts (folds)
- Fold 1: Train on folds 2,3,4,5 → Test on fold 1
- Fold 2: Train on folds 1,3,4,5 → Test on fold 2
- ... and so on
- Report: Mean ± Standard Deviation

In [ ]:
print("K-FOLD CROSS-VALIDATION")
print("="*60)

# Initialize model and CV
model = LogisticRegression(random_state=RANDOM_STATE)
kfold = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Perform cross-validation
cv_results = cross_validate(
    model, X_balanced, y_balanced,
    cv=kfold,
    scoring=['accuracy', 'precision', 'recall', 'f1'],
    return_train_score=True
)

print("5-Fold Cross-Validation Results:")
print(f"\nFold-by-fold accuracy:")
for fold, score in enumerate(cv_results['test_accuracy'], 1):
    print(f"  Fold {fold}: {score:.4f}")

print("\nSummary Statistics:")
metrics = ['accuracy', 'precision', 'recall', 'f1']
for metric in metrics:
    mean = cv_results[f'test_{metric}'].mean()
    std = cv_results[f'test_{metric}'].std()
    print(f"  {metric.capitalize()}: {mean:.4f} (± {std:.4f})")

print("\n✅ CV gives us mean ± std - much more informative!")

## Step 5: Visualize K-Fold Cross-Validation

Let's visualize how K-Fold splits the data and the performance across folds.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Fold visualization
fold_viz_data = []
for fold, (train_idx, test_idx) in enumerate(kfold.split(X_balanced), 1):
    fold_viz_data.append({
        'Fold': fold,
        'Train Size': len(train_idx),
        'Test Size': len(test_idx)
    })

fold_df = pd.DataFrame(fold_viz_data)
x = np.arange(len(fold_df))
width = 0.35
axes[0, 0].bar(x - width/2, fold_df['Train Size'], width, label='Train', color='#2ecc71')
axes[0, 0].bar(x + width/2, fold_df['Test Size'], width, label='Test', color='#e74c3c')
axes[0, 0].set_xlabel('Fold', fontsize=11)
axes[0, 0].set_ylabel('Number of Samples', fontsize=11)
axes[0, 0].set_title('K-Fold Split Sizes', fontsize=12, fontweight='bold')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels([f'Fold {i}' for i in range(1, 6)])
axes[0, 0].legend()

# 2. Performance across folds
folds = range(1, 6)
axes[0, 1].plot(folds, cv_results['test_accuracy'], 'o-', label='Test Accuracy', linewidth=2)
axes[0, 1].plot(folds, cv_results['train_accuracy'], 's--', label='Train Accuracy', linewidth=2)
axes[0, 1].axhline(y=cv_results['test_accuracy'].mean(), color='r', linestyle=':', 
                   label=f'Mean: {cv_results["test_accuracy"].mean():.4f}')
axes[0, 1].set_xlabel('Fold', fontsize=11)
axes[0, 1].set_ylabel('Accuracy', fontsize=11)
axes[0, 1].set_title('Accuracy Across Folds', fontsize=12, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# 3. Box plot of all metrics
metrics_data = [cv_results[f'test_{m}'] for m in metrics]
bp = axes[1, 0].boxplot(metrics_data, labels=[m.capitalize() for m in metrics],
                        patch_artist=True)
for patch, color in zip(bp['boxes'], ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']):
    patch.set_facecolor(color)
axes[1, 0].set_ylabel('Score', fontsize=11)
axes[1, 0].set_title('Score Distribution Across Folds', fontsize=12, fontweight='bold')
axes[1, 0].grid(alpha=0.3)

# 4. Mean scores with error bars
means = [cv_results[f'test_{m}'].mean() for m in metrics]
stds = [cv_results[f'test_{m}'].std() for m in metrics]
axes[1, 1].bar(range(len(metrics)), means, yerr=stds, capsize=5,
              color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12'])
axes[1, 1].set_xticks(range(len(metrics)))
axes[1, 1].set_xticklabels([m.capitalize() for m in metrics])
axes[1, 1].set_ylabel('Score', fontsize=11)
axes[1, 1].set_title('Mean Scores with Standard Deviation', fontsize=12, fontweight='bold')
axes[1, 1].set_ylim(0, 1.0)
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("📊 Visualizations show how performance varies across folds")

## Step 6: Stratified K-Fold for Imbalanced Data

When dealing with imbalanced classes, regular K-Fold might create folds with very different class distributions. Stratified K-Fold maintains the same class ratio in each fold.

> **💡 AI Assistant Prompt:**
> 
> "Explain why Stratified K-Fold is important for imbalanced datasets. Show code comparing regular K-Fold vs Stratified K-Fold on a dataset with 10% minority class."

In [ ]:
print("REGULAR K-FOLD vs STRATIFIED K-FOLD")
print("="*60)

# Regular K-Fold
print("\n1. Regular K-Fold on Imbalanced Data:")
regular_kfold = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

regular_fold_distributions = []
for fold, (train_idx, test_idx) in enumerate(regular_kfold.split(X_imbalanced), 1):
    test_class_dist = (y_imbalanced[test_idx] == 1).sum() / len(test_idx)
    regular_fold_distributions.append(test_class_dist * 100)
    print(f"   Fold {fold}: {test_class_dist*100:.2f}% minority class in test set")

# Stratified K-Fold
print("\n2. Stratified K-Fold on Imbalanced Data:")
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

stratified_fold_distributions = []
for fold, (train_idx, test_idx) in enumerate(stratified_kfold.split(X_imbalanced, y_imbalanced), 1):
    test_class_dist = (y_imbalanced[test_idx] == 1).sum() / len(test_idx)
    stratified_fold_distributions.append(test_class_dist * 100)
    print(f"   Fold {fold}: {test_class_dist*100:.2f}% minority class in test set")

print(f"\nRegular K-Fold: Std = {np.std(regular_fold_distributions):.2f}%")
print(f"Stratified K-Fold: Std = {np.std(stratified_fold_distributions):.2f}%")

# Visualize the difference
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution comparison
folds = range(1, 6)
axes[0].plot(folds, regular_fold_distributions, 'o-', label='Regular K-Fold', 
            linewidth=2, markersize=8)
axes[0].plot(folds, stratified_fold_distributions, 's-', label='Stratified K-Fold', 
            linewidth=2, markersize=8)
axes[0].axhline(y=10, color='r', linestyle='--', label='True Distribution (10%)')
axes[0].set_xlabel('Fold', fontsize=12)
axes[0].set_ylabel('Minority Class %', fontsize=12)
axes[0].set_title('Class Distribution Across Folds', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Box plot comparison
data_to_plot = [regular_fold_distributions, stratified_fold_distributions]
bp = axes[1].boxplot(data_to_plot, labels=['Regular K-Fold', 'Stratified K-Fold'],
                     patch_artist=True)
bp['boxes'][0].set_facecolor('#e74c3c')
bp['boxes'][1].set_facecolor('#2ecc71')
axes[1].axhline(y=10, color='r', linestyle='--', label='True Distribution (10%)')
axes[1].set_ylabel('Minority Class %', fontsize=12)
axes[1].set_title('Distribution Stability', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ Stratified K-Fold maintains consistent class distribution!")
print("   Use it whenever you have imbalanced classes.")

## Step 7: Compare Performance - Regular vs Stratified

Let's see if Stratified K-Fold gives more reliable performance estimates on imbalanced data.

In [ ]:
print("PERFORMANCE COMPARISON: Regular vs Stratified K-Fold")
print("="*60)

model = LogisticRegression(random_state=RANDOM_STATE)

# Regular K-Fold results
regular_scores = cross_val_score(model, X_imbalanced, y_imbalanced, 
                                 cv=regular_kfold, scoring='f1')

# Stratified K-Fold results
stratified_scores = cross_val_score(model, X_imbalanced, y_imbalanced, 
                                    cv=stratified_kfold, scoring='f1')

print("Regular K-Fold F1 Scores:")
for fold, score in enumerate(regular_scores, 1):
    print(f"  Fold {fold}: {score:.4f}")
print(f"  Mean: {regular_scores.mean():.4f} (± {regular_scores.std():.4f})")

print("\nStratified K-Fold F1 Scores:")
for fold, score in enumerate(stratified_scores, 1):
    print(f"  Fold {fold}: {score:.4f}")
print(f"  Mean: {stratified_scores.mean():.4f} (± {stratified_scores.std():.4f})")

# Visualize
plt.figure(figsize=(12, 5))

# Side-by-side comparison
x = np.arange(5)
width = 0.35
plt.bar(x - width/2, regular_scores, width, label='Regular K-Fold', color='#e74c3c', alpha=0.7)
plt.bar(x + width/2, stratified_scores, width, label='Stratified K-Fold', color='#2ecc71', alpha=0.7)

plt.axhline(y=regular_scores.mean(), color='red', linestyle='--', alpha=0.5)
plt.axhline(y=stratified_scores.mean(), color='green', linestyle='--', alpha=0.5)

plt.xlabel('Fold', fontsize=12)
plt.ylabel('F1-Score', fontsize=12)
plt.title('F1-Score Comparison: Regular vs Stratified K-Fold', fontsize=14, fontweight='bold')
plt.xticks(x, [f'Fold {i}' for i in range(1, 6)])
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nStability Improvement:")
print(f"  Regular K-Fold std: {regular_scores.std():.4f}")
print(f"  Stratified K-Fold std: {stratified_scores.std():.4f}")
print(f"  Reduction in variance: {(1 - stratified_scores.std()/regular_scores.std())*100:.1f}%")

print("\n✅ Stratified K-Fold provides more stable estimates!")

## Step 8: Time Series Cross-Validation

For time-series data, we CANNOT use regular K-Fold because it would train on future data to predict past data (data leakage!). Time Series CV uses forward-chaining.

**How Time Series CV Works:**
- Split 1: Train on [0:200] → Test on [200:400]
- Split 2: Train on [0:400] → Test on [400:600]
- Split 3: Train on [0:600] → Test on [600:800]
- Split 4: Train on [0:800] → Test on [800:1000]

Training set grows progressively, always using past to predict future.

> **💡 AI Assistant Prompt:**
> 
> "Explain why regular cross-validation causes data leakage in time-series problems. Show code implementing Time Series Split and visualize how it differs from K-Fold."

In [ ]:
print("TIME SERIES CROSS-VALIDATION")
print("="*60)

# Time Series Split
tscv = TimeSeriesSplit(n_splits=5)

print("Time Series CV Split Information:")
for fold, (train_idx, test_idx) in enumerate(tscv.split(X_timeseries), 1):
    print(f"\nFold {fold}:")
    print(f"  Train indices: {train_idx[0]} to {train_idx[-1]} (n={len(train_idx)})")
    print(f"  Test indices:  {test_idx[0]} to {test_idx[-1]} (n={len(test_idx)})")
    print(f"  Train dates: {time_index[train_idx[0]].date()} to {time_index[train_idx[-1]].date()}")
    print(f"  Test dates:  {time_index[test_idx[0]].date()} to {time_index[test_idx[-1]].date()}")

# Visualize Time Series Split
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# 1. Visualization of splits
colors = plt.cm.viridis(np.linspace(0, 1, 5))
for fold, (train_idx, test_idx) in enumerate(tscv.split(X_timeseries)):
    # Train indices
    axes[0].barh(fold, len(train_idx), left=train_idx[0], 
                height=0.4, color=colors[fold], alpha=0.6, label=f'Fold {fold+1} Train')
    # Test indices
    axes[0].barh(fold, len(test_idx), left=test_idx[0], 
                height=0.4, color=colors[fold], alpha=1.0, edgecolor='black', linewidth=2)

axes[0].set_yticks(range(5))
axes[0].set_yticklabels([f'Fold {i}' for i in range(1, 6)])
axes[0].set_xlabel('Sample Index', fontsize=12)
axes[0].set_title('Time Series Cross-Validation Splits (Training=light, Testing=dark)', 
                 fontsize=14, fontweight='bold')
axes[0].grid(alpha=0.3)

# 2. Train/Test set sizes
train_sizes = []
test_sizes = []
for train_idx, test_idx in tscv.split(X_timeseries):
    train_sizes.append(len(train_idx))
    test_sizes.append(len(test_idx))

x = np.arange(5)
width = 0.35
axes[1].bar(x - width/2, train_sizes, width, label='Train Size', color='#2ecc71')
axes[1].bar(x + width/2, test_sizes, width, label='Test Size', color='#e74c3c')
axes[1].set_xlabel('Fold', fontsize=12)
axes[1].set_ylabel('Number of Samples', fontsize=12)
axes[1].set_title('Train/Test Sizes Across Folds (Note: Train Size Grows)', 
                 fontsize=14, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'Fold {i}' for i in range(1, 6)])
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ Time Series CV prevents data leakage by always training on past!")

## Step 9: Compare Regular K-Fold vs Time Series CV (Data Leakage Demo)

In [ ]:
print("DEMONSTRATING DATA LEAKAGE")
print("="*60)

model = LogisticRegression(random_state=RANDOM_STATE)

# Regular K-Fold (WRONG for time series - causes leakage)
regular_kfold_ts = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
regular_scores_ts = cross_val_score(model, X_timeseries, y_timeseries, 
                                   cv=regular_kfold_ts, scoring='accuracy')

# Time Series CV (CORRECT - no leakage)
tscv_scores = cross_val_score(model, X_timeseries, y_timeseries, 
                              cv=tscv, scoring='accuracy')

print("Regular K-Fold (with data leakage):")
print(f"  Mean accuracy: {regular_scores_ts.mean():.4f} (± {regular_scores_ts.std():.4f})")
print("  ⚠️ OVERLY OPTIMISTIC - trains on future to predict past!")

print("\nTime Series CV (correct):")
print(f"  Mean accuracy: {tscv_scores.mean():.4f} (± {tscv_scores.std():.4f})")
print("  ✅ REALISTIC - only trains on past data")

print(f"\nInflation due to data leakage: {(regular_scores_ts.mean() - tscv_scores.mean())*100:.2f} percentage points")

# Visualize the difference
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scores comparison
folds = range(1, 6)
axes[0].plot(folds, regular_scores_ts, 'o-', label='Regular K-Fold (WRONG)', 
            linewidth=2, markersize=8, color='#e74c3c')
axes[0].plot(folds, tscv_scores, 's-', label='Time Series CV (CORRECT)', 
            linewidth=2, markersize=8, color='#2ecc71')
axes[0].set_xlabel('Fold', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Performance Comparison', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Box plot
data_to_plot = [regular_scores_ts, tscv_scores]
bp = axes[1].boxplot(data_to_plot, labels=['K-Fold\n(Leakage)', 'Time Series CV\n(Correct)'],
                     patch_artist=True)
bp['boxes'][0].set_facecolor('#e74c3c')
bp['boxes'][1].set_facecolor('#2ecc71')
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Score Distribution', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🚨 KEY LESSON: Always use Time Series CV for temporal data!")
print("   Regular K-Fold will give misleadingly high scores.")

## Step 10: Interpreting Cross-Validation Results

Understanding what mean ± standard deviation tells us about model stability.

In [ ]:
print("INTERPRETING CV RESULTS: Mean ± Standard Deviation")
print("="*60)

# Train three different models and compare stability
models = {
    'Logistic Regression': LogisticRegression(random_state=RANDOM_STATE),
    'Random Forest (n=10)': RandomForestClassifier(n_estimators=10, random_state=RANDOM_STATE),
    'Random Forest (n=100)': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
}

results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_balanced, y_balanced, 
                            cv=5, scoring='accuracy')
    results[name] = {
        'scores': scores,
        'mean': scores.mean(),
        'std': scores.std()
    }
    print(f"\n{name}:")
    print(f"  Scores: {[f'{s:.4f}' for s in scores]}")
    print(f"  Mean: {results[name]['mean']:.4f}")
    print(f"  Std:  {results[name]['std']:.4f}")
    
    # Interpretation
    if results[name]['std'] < 0.01:
        stability = "Very stable ✅"
    elif results[name]['std'] < 0.03:
        stability = "Stable ✅"
    elif results[name]['std'] < 0.05:
        stability = "Moderate ⚠️"
    else:
        stability = "Unstable 🚨"
    print(f"  Stability: {stability}")

# Visualize stability
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plots
data_to_plot = [results[name]['scores'] for name in models.keys()]
bp = axes[0].boxplot(data_to_plot, labels=list(models.keys()), patch_artist=True)
for patch, color in zip(bp['boxes'], ['#3498db', '#e74c3c', '#2ecc71']):
    patch.set_facecolor(color)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Model Stability Comparison', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(alpha=0.3)

# Mean with error bars
model_names = list(models.keys())
means = [results[name]['mean'] for name in model_names]
stds = [results[name]['std'] for name in model_names]
x = np.arange(len(model_names))

axes[1].bar(x, means, yerr=stds, capsize=10, 
           color=['#3498db', '#e74c3c', '#2ecc71'], alpha=0.7)
axes[1].set_xticks(x)
axes[1].set_xticklabels(model_names, rotation=45, ha='right')
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Mean ± Standard Deviation', fontsize=14, fontweight='bold')
axes[1].set_ylim(0.7, 1.0)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 INTERPRETATION GUIDE:")
print("  • Low std (<0.02): Model is very stable across different data subsets")
print("  • High std (>0.05): Model is sensitive to data - might overfit")
print("  • Compare models: Choose high mean AND low std for best generalization")

## Step 11: Common Mistakes to Avoid

Let's highlight common cross-validation pitfalls.

In [ ]:
print("COMMON CROSS-VALIDATION MISTAKES")
print("="*60)

print("\n❌ MISTAKE 1: Using K-Fold on imbalanced data")
print("   Problem: Folds might have very different class distributions")
print("   Solution: Use StratifiedKFold instead")

print("\n❌ MISTAKE 2: Using K-Fold on time-series data")
print("   Problem: Trains on future to predict past (data leakage)")
print("   Solution: Use TimeSeriesSplit instead")

print("\n❌ MISTAKE 3: Data preprocessing before CV split")
print("   Example: Scaling entire dataset before CV")
print("   Problem: Test data information leaks into training")
print("   Solution: Fit scaler only on training folds, transform test folds")

print("\n❌ MISTAKE 4: Choosing too many folds for small datasets")
print("   Problem: Very small test sets, high variance")
print("   Solution: Use 5-fold for datasets < 1000, 10-fold for larger")

print("\n❌ MISTAKE 5: Reporting only mean without std")
print("   Problem: Hides model instability")
print("   Solution: Always report mean ± std")

print("\n❌ MISTAKE 6: Using CV scores for final model evaluation")
print("   Problem: CV is for model selection, not final evaluation")
print("   Solution: Use separate held-out test set for final evaluation")

print("\n✅ BEST PRACTICES:")
print("   1. Choose CV strategy based on data type")
print("   2. Always report mean ± std")
print("   3. Use stratified CV for imbalanced data")
print("   4. Use time series CV for temporal data")
print("   5. Fit preprocessing inside CV loop")
print("   6. Keep separate test set for final evaluation")

## Summary: What You Learned

### Key Concepts:

1. **Why Cross-Validation**: Single train/test split can give lucky/unlucky results. CV provides more reliable estimates with mean ± std.

2. **K-Fold CV**: Divides data into K folds, each serving as test set once. Gives K performance scores to average.

3. **Stratified K-Fold**: Maintains class distribution in each fold. Essential for imbalanced datasets.

4. **Time Series CV**: Uses forward-chaining to prevent data leakage. Training set grows, always trains on past to predict future.

5. **Interpreting Results**: Mean tells you average performance, std tells you stability. Look for high mean AND low std.

### When to Use Each:

| Data Type | CV Strategy | Why |
|-----------|-------------|-----|
| Balanced classes | K-Fold | Simple and effective |
| Imbalanced classes | Stratified K-Fold | Maintains class ratio |
| Time-series | Time Series Split | Prevents data leakage |
| Small dataset (< 100) | LOOCV | Maximizes training data |

### Quiz Preparation:

You should now be able to answer:
- Why is CV better than single split? (M4-S3-Q01)
- How much data is used for training in 5-fold CV? (M4-S3-Q02)
- When to use Stratified K-Fold? (M4-S3-Q03)
- Why not use regular CV for time-series? (M4-S3-Q04)
- What does high std in CV scores mean? (M4-S3-Q06)

### Next Steps:

1. Practice on your own datasets
2. Try different numbers of folds (3, 5, 10)
3. Combine CV with hyperparameter tuning (next session!)
4. Always use appropriate CV for your data type

**Remember**: Cross-validation is not a replacement for a final test set. Use CV to compare models and select hyperparameters, then evaluate the final model on a held-out test set!

> **💡 AI Assistant Challenge:**
> 
> "I have a dataset of 5000 customer transactions from the last year, with 2% fraud rate. I want to build a fraud detection model. What cross-validation strategy should I use and why? Write code to implement it."

**Great work! You now understand cross-validation and can apply it correctly to any ML problem!** 🎉